# Notebook 6 — Final Grounded LLM Explanation (30 Samples)

**Pipeline:** ResNet50 → Trust/Uncertainty → Grad-CAM → RAG → Qwen → Structured Explanation

This is the final lightweight version of Notebook 6.

### Goal
Generate grounded LLM explanations for **30 balanced test samples** and save every result to Google Drive.

### Important design decisions
- The complete 1,166-image test set has already been evaluated by ResNet50, Trust/Uncertainty and Grad-CAM in Notebook 4/3.
- This notebook does **not** rerun the classifier, Grad-CAM, or the full 1,166-sample LLM experiment.
- Exactly 30 balanced samples are selected: **8 Healthy, 8 Scab, 7 Black Rot, 7 Cedar Apple Rust**.
- Notebook 5 RAG artifacts are reused.
- `knowledge_chunks.csv` is used as FAISS-aligned metadata.
- MiniLM is used **on CPU** only for the four cached disease queries, so it does not consume T4 VRAM.
- Qwen is loaded as **Qwen2.5-3B-Instruct, 4-bit NF4**, which is substantially more suitable for a T4 than the previous 7B setup.
- Every successful explanation is saved immediately, so the run can resume after interruption.

## Expected Google Drive structure

```text
XAI_Project/
└── results/
    ├── trust_uncertainty/
    │   └── trust_uncertainty_test_report.csv
    │
    ├── rag/
    │   ├── apple_disease_faiss.index
    │   ├── knowledge_chunks.csv
    │   ├── knowledge_documents.csv
    │   ├── knowledge_embeddings.npy
    │   └── rag_config.json
    │
    └── llm_explanations/
        └── llm_30_samples/
            ├── per_sample/
            │   ├── test_XXXXX.json
            │   └── ...
            ├── llm_explanations_30_samples.jsonl
            ├── llm_explanations_30_samples.csv
            ├── rag_retrieval_cache.json
            ├── llm_30_config.json
            ├── smoke_test_result.json
            └── llm_errors_30_samples.jsonl
```

In [ ]:
# 1. INSTALL ONLY REQUIRED PACKAGES
!pip -q install -U transformers accelerate bitsandbytes faiss-cpu tqdm
print("Dependencies installed.")

In [ ]:
# 2. IMPORTS + FRESH GPU CHECK
import os
import gc
import json
import re
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

print("NumPy:", np.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required for Qwen inference.")

print("GPU:", torch.cuda.get_device_name(0))

free_vram, total_vram = torch.cuda.mem_get_info()
print(f"Free VRAM:  {free_vram / 1024**3:.2f} GB")
print(f"Total VRAM: {total_vram / 1024**3:.2f} GB")

In [ ]:
# 3. MOUNT GOOGLE DRIVE + DEFINE PATHS
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_ROOT = DRIVE_ROOT / "XAI_Project"
RESULTS_ROOT = PROJECT_ROOT / "results"

TRUST_DIR = RESULTS_ROOT / "trust_uncertainty"
RAG_DIR = RESULTS_ROOT / "rag"
LLM_ROOT = RESULTS_ROOT / "llm_explanations"
LLM_30_DIR = LLM_ROOT / "llm_30_samples"

PER_SAMPLE_DIR = LLM_30_DIR / "per_sample"

LLM_30_DIR.mkdir(parents=True, exist_ok=True)
PER_SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("30-sample output:", LLM_30_DIR)

In [ ]:
# 4. LOCATE NOTEBOOK 4 + NOTEBOOK 5 ARTIFACTS
TRUST_REPORT = TRUST_DIR / "trust_uncertainty_test_report.csv"

RAG_INDEX = RAG_DIR / "apple_disease_faiss.index"
RAG_CHUNKS = RAG_DIR / "knowledge_chunks.csv"

# Optional files retained for documentation/reference.
RAG_DOCUMENTS = RAG_DIR / "knowledge_documents.csv"
RAG_EMBEDDINGS = RAG_DIR / "knowledge_embeddings.npy"
RAG_CONFIG = RAG_DIR / "rag_config.json"

required = {
    "Notebook 4 trust report": TRUST_REPORT,
    "Notebook 5 FAISS index": RAG_INDEX,
    "Notebook 5 knowledge chunks": RAG_CHUNKS,
}

for name, path in required.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")

print("All required artifacts found.")
print("Trust report:", TRUST_REPORT)
print("FAISS:", RAG_INDEX)
print("Knowledge chunks:", RAG_CHUNKS)

In [ ]:
# 5. LOAD + VALIDATE NOTEBOOK 4 REPORT

trust_df = pd.read_csv(TRUST_REPORT)

CLASS_NAMES = {
    0: "Apple Healthy",
    1: "Apple Scab",
    2: "Apple Black Rot",
    3: "Apple Cedar Apple Rust",
}

REQUIRED_COLUMNS = [
    "image_index",
    "true_label",
    "true_class",
    "predicted_label",
    "predicted_class",
    "correct",
    "confidence",
    "entropy",
    "margin",
    "trust_score",
    "trust_level",
    "gradcam_high_activation_area_percent",
]

missing = [c for c in REQUIRED_COLUMNS if c not in trust_df.columns]

if missing:
    raise ValueError(f"Missing Notebook 4 columns: {missing}")

# Robustly detect probability columns instead of assuming a fixed naming pattern.
def get_probability_columns(df):
    candidates = [
        [f"prob_{i}" for i in range(4)],
        [f"probability_{i}" for i in range(4)],
        [
            "prob_0_Apple_Healthy",
            "prob_1_Apple_Scab",
            "prob_2_Apple_Black_Rot",
            "prob_3_Apple_Cedar_Apple_Rust",
        ],
    ]

    for cols in candidates:
        if all(c in df.columns for c in cols):
            return cols

    prob_cols = [
        c for c in df.columns
        if "prob" in str(c).lower()
    ]

    if len(prob_cols) >= 4:
        return prob_cols[:4]

    raise ValueError(
        "Could not identify the 4 classifier probability columns."
    )

PROB_COLS = get_probability_columns(trust_df)

work_df = trust_df.copy()

work_df["sample_id"] = [
    f"test_{int(i):05d}" for i in work_df["image_index"]
]

work_df["high_activation_area_percent"] = (
    work_df["gradcam_high_activation_area_percent"]
)

def probabilities(row):
    return {
        CLASS_NAMES[i]: float(row[PROB_COLS[i]])
        for i in range(4)
    }

print("Trust report shape:", work_df.shape)
print("Probability columns:", PROB_COLS)
display(
    work_df[
        [
            "sample_id",
            "true_class",
            "predicted_class",
            "confidence",
            "trust_score",
            "trust_level",
        ]
    ].head()
)

In [ ]:
# 6. SELECT EXACTLY 30 BALANCED TEST SAMPLES

SAMPLES_PER_CLASS = {
    0: 8,  # Apple Healthy
    1: 8,  # Apple Scab
    2: 7,  # Apple Black Rot
    3: 7,  # Apple Cedar Apple Rust
}

selected_parts = []

for label, count in SAMPLES_PER_CLASS.items():
    class_df = work_df[
        work_df["predicted_label"].astype(int) == label
    ].copy()

    class_df = class_df.sort_values("sample_id").head(count)

    if len(class_df) < count:
        raise ValueError(
            f"Only {len(class_df)} samples available for "
            f"{CLASS_NAMES[label]}, expected {count}."
        )

    selected_parts.append(class_df)

llm_df = pd.concat(
    selected_parts,
    ignore_index=True
)

assert len(llm_df) == 30

print("==========================================")
print("30-SAMPLE LLM EXPERIMENT")
print("==========================================")
print("Total selected:", len(llm_df))

print("\nClass distribution:")
print(
    llm_df["predicted_label"]
    .astype(int)
    .map(CLASS_NAMES)
    .value_counts()
)

display(
    llm_df[
        [
            "sample_id",
            "true_class",
            "predicted_class",
            "confidence",
            "trust_score",
            "trust_level",
        ]
    ]
)

In [ ]:
# 7. LOAD FAISS INDEX + CORRECT NOTEBOOK 5 METADATA

import faiss

faiss_index = faiss.read_index(str(RAG_INDEX))
knowledge_chunks = pd.read_csv(RAG_CHUNKS)

print("FAISS dimension:", faiss_index.d)
print("FAISS vectors:", faiss_index.ntotal)
print("Knowledge chunk rows:", len(knowledge_chunks))
print("Knowledge chunk columns:", list(knowledge_chunks.columns))

if faiss_index.d != 384:
    raise ValueError(
        f"Expected FAISS dimension 384, found {faiss_index.d}"
    )

if faiss_index.ntotal != len(knowledge_chunks):
    raise ValueError(
        "FAISS index and knowledge_chunks.csv are not aligned: "
        f"{faiss_index.ntotal} vectors vs {len(knowledge_chunks)} rows."
    )

# Normalize common Notebook 5 column names.
def find_column(df, candidates, required=True):
    lower = {str(c).lower(): c for c in df.columns}

    for candidate in candidates:
        if candidate.lower() in lower:
            return lower[candidate.lower()]

    for c in df.columns:
        lc = str(c).lower()
        for candidate in candidates:
            if candidate.lower() in lc:
                return c

    if required:
        raise ValueError(
            f"Could not find one of {candidates} in knowledge_chunks.csv"
        )

    return None

TEXT_COL = find_column(
    knowledge_chunks,
    ["text", "content", "chunk", "document", "page_content"]
)

TITLE_COL = find_column(
    knowledge_chunks,
    ["title", "name", "source_title"],
    required=False
)

SOURCE_COL = find_column(
    knowledge_chunks,
    ["source", "source_url", "url", "reference", "citation"],
    required=False
)

print("Text column:", TEXT_COL)
print("Title column:", TITLE_COL)
print("Source column:", SOURCE_COL)

In [ ]:
# 8. LOAD MINILM ON CPU ONLY
# This is intentional: it keeps T4 VRAM available for Qwen.

from transformers import AutoTokenizer, AutoModel

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

embedding_tokenizer = AutoTokenizer.from_pretrained(
    EMBEDDING_MODEL_NAME
)

embedding_model = AutoModel.from_pretrained(
    EMBEDDING_MODEL_NAME
)

embedding_device = torch.device("cpu")
embedding_model = embedding_model.to(embedding_device)
embedding_model.eval()

print("MiniLM loaded on:", embedding_device)
print("Embedding dimension: 384")

In [ ]:
# 9. MINILM QUERY ENCODER + RAG RETRIEVAL

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(
        token_embeddings.size()
    ).float()

    return torch.sum(
        token_embeddings * mask,
        dim=1
    ) / torch.clamp(
        mask.sum(dim=1),
        min=1e-9
    )

def encode_query(text):
    tokens = embedding_tokenizer(
        [text],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    tokens = {
        k: v.to(embedding_device)
        for k, v in tokens.items()
    }

    with torch.no_grad():
        output = embedding_model(**tokens)
        pooled = mean_pooling(
            output,
            tokens["attention_mask"]
        )
        pooled = F.normalize(
            pooled,
            p=2,
            dim=1
        )

    return pooled.cpu().numpy().astype("float32")

def retrieve_rag(query, top_k=5):
    query_vector = encode_query(query)

    scores, indices = faiss_index.search(
        query_vector,
        top_k
    )

    results = []

    for score, idx in zip(
        scores[0],
        indices[0]
    ):
        if idx < 0:
            continue

        row = knowledge_chunks.iloc[int(idx)]

        title = (
            str(row[TITLE_COL])
            if TITLE_COL is not None
            else "Knowledge document"
        )

        source = (
            str(row[SOURCE_COL])
            if SOURCE_COL is not None
            else "Source metadata unavailable"
        )

        text = str(row[TEXT_COL])

        results.append({
            "rank": len(results) + 1,
            "score": float(score),
            "title": title,
            "source": source,
            "text": text,
        })

    return results

# Verify embedding dimension and retrieval.
test_vector = encode_query(
    "Apple scab symptoms disease cycle and management"
)

assert test_vector.shape == (1, 384)

test_results = retrieve_rag(
    "Apple scab symptoms disease cycle and management",
    top_k=5
)

print("Query embedding:", test_vector.shape)
print("Retrieved chunks:", len(test_results))

display(pd.DataFrame(test_results))

In [ ]:
# 10. CACHE RAG RETRIEVAL FOR THE FOUR DISEASE CLASSES

TOP_K = 5

RAG_CACHE_FILE = (
    LLM_30_DIR /
    "rag_retrieval_cache.json"
)

RAG_QUERIES = {
    "Apple Healthy":
        "Apple healthy leaf characteristics and normal appearance",

    "Apple Scab":
        "Apple scab symptoms disease cycle causes and management",

    "Apple Black Rot":
        "Apple black rot symptoms disease cycle causes and management",

    "Apple Cedar Apple Rust":
        "Apple cedar apple rust symptoms disease cycle causes and management",
}

rag_cache = None

if RAG_CACHE_FILE.exists():
    try:
        with open(
            RAG_CACHE_FILE,
            "r",
            encoding="utf-8"
        ) as f:
            loaded_cache = json.load(f)

        if set(loaded_cache.keys()) == set(RAG_QUERIES.keys()):
            rag_cache = loaded_cache
            print("Existing RAG cache loaded.")

    except Exception as e:
        print("Existing cache could not be loaded:", e)

if rag_cache is None:

    print("Creating RAG cache...")

    rag_cache = {}

    for disease, query in RAG_QUERIES.items():

        print("Retrieving:", disease)

        rag_cache[disease] = retrieve_rag(
            query,
            top_k=TOP_K
        )

    with open(
        RAG_CACHE_FILE,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            rag_cache,
            f,
            indent=2,
            ensure_ascii=False
        )

print("\nRAG cache summary:")

for disease in CLASS_NAMES.values():

    print(
        disease,
        "→",
        len(rag_cache[disease]),
        "chunks"
    )

# Release MiniLM BEFORE loading Qwen.
del embedding_model
del embedding_tokenizer

gc.collect()

print("\nMiniLM released from memory.")

free_vram, total_vram = torch.cuda.mem_get_info()

print(
    f"Free T4 VRAM before Qwen: "
    f"{free_vram / 1024**3:.2f} GB"
)

## Qwen explanation model

The previous Notebook 6 attempted Qwen2.5-7B-Instruct. On the available T4, the runtime repeatedly had insufficient usable VRAM.

For this final notebook, **Qwen2.5-3B-Instruct** is used in 4-bit NF4 quantization.

This does not change the XAI architecture. Qwen is only the natural-language explanation layer; the classifier, trust score, Grad-CAM and RAG evidence remain the primary structured evidence.

In [ ]:
# 11. LOAD QWEN2.5-3B-INSTRUCT — 4-BIT NF4

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

# Clear any remaining CUDA cache before model loading.
gc.collect()
torch.cuda.empty_cache()

LLM_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading:", LLM_MODEL_NAME)

llm_tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME,
    trust_remote_code=True,
)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    trust_remote_code=True,
)

llm_model.eval()

print("\n==========================================")
print("QWEN LOADED SUCCESSFULLY")
print("==========================================")

print("Model:", LLM_MODEL_NAME)
print(
    "Device:",
    next(llm_model.parameters()).device
)

free_vram, total_vram = torch.cuda.mem_get_info()

print(
    f"Free VRAM:  {free_vram / 1024**3:.2f} GB"
)

print(
    f"Total VRAM: {total_vram / 1024**3:.2f} GB"
)

In [ ]:
def generate_qwen(prompt, max_new_tokens=160):

    messages = [
        {
            "role": "system",
            "content": (
                "Return exactly one valid JSON object. "
                "Use extremely short answers. "
                "No Markdown. "
                "No URLs. "
                "No text outside JSON."
            ),
        },
        {
            "role": "user",
            "content": prompt,
        },
    ]

    text = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = llm_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=6000,
    )

    inputs = {
        k: v.to(llm_model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        output = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=1,
            use_cache=True,
            pad_token_id=llm_tokenizer.eos_token_id,
            eos_token_id=llm_tokenizer.eos_token_id,
        )

    new_tokens = output[
        0,
        inputs["input_ids"].shape[1]:
    ]

    response = llm_tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
    ).strip()

    return response

In [ ]:
# 13. ROBUST JSON EXTRACTION + VALIDATION

EXPECTED_KEYS = [
    "summary",
    "prediction_explanation",
    "visual_evidence",
    "knowledge_evidence",
    "uncertainty_note",
    "human_review_recommendation",
    "source_references",
]

def clean_json_response(response):

    if response is None:
        raise ValueError("Empty Qwen response.")

    response = str(response).strip()

    if not response:
        raise ValueError("Empty Qwen response.")

    # Remove markdown fences
    response = re.sub(
        r"```json\s*",
        "",
        response,
        flags=re.IGNORECASE
    )

    response = re.sub(
        r"```\s*",
        "",
        response
    )

    response = response.strip()

    # --------------------------------------------------
    # 1. Try normal JSON first
    # --------------------------------------------------

    try:

        obj = json.loads(response)

        if isinstance(obj, dict):
            return obj

    except Exception:
        pass

    # --------------------------------------------------
    # 2. Try extracting a complete JSON object
    # --------------------------------------------------

    decoder = json.JSONDecoder()

    for match in re.finditer(r"\{", response):

        start = match.start()

        try:

            obj, end = decoder.raw_decode(
                response[start:]
            )

            if isinstance(obj, dict):
                return obj

        except json.JSONDecodeError:
            continue

    # --------------------------------------------------
    # 3. Attempt recovery of truncated JSON
    # --------------------------------------------------

    start = response.find("{")

    if start == -1:
        raise ValueError(
            "Qwen response did not contain a JSON object."
        )

    partial = response[start:].strip()

    # Remove incomplete final string
    # Example:
    # "source_references": ["abc
    partial = re.sub(
        r',?\s*"[^"]*"\s*:\s*"[^"]*$',
        "",
        partial
    )

    # Remove incomplete array/string
    partial = re.sub(
        r',?\s*"[^"]*"\s*:\s*\[[^\]]*$',
        "",
        partial
    )

    # Remove dangling comma
    partial = re.sub(
        r',\s*$',
        "",
        partial
    )

    # Close any remaining open string
    if partial.count('"') % 2 != 0:
        partial += '"'

    # Close arrays
    open_arrays = (
        partial.count("[")
        - partial.count("]")
    )

    for _ in range(max(0, open_arrays)):
        partial += "]"

    # Close objects
    open_objects = (
        partial.count("{")
        - partial.count("}")
    )

    for _ in range(max(0, open_objects)):
        partial += "}"

    try:

        obj = json.loads(partial)

        if isinstance(obj, dict):
            return obj

    except Exception:
        pass

    raise ValueError(
        "Qwen response could not be parsed or safely repaired."
    )

def validate_json(obj):

    required_keys = [
        "summary",
        "prediction_explanation",
        "visual_evidence",
        "knowledge_evidence",
        "uncertainty_note",
        "human_review_recommendation",
        "source_references",
    ]

    for key in required_keys:

        if key not in obj:
            raise ValueError(
                f"Missing required key: {key}"
            )

    if not isinstance(
        obj["source_references"],
        list
    ):
        raise ValueError(
            "source_references must be a list."
        )

    for source_id in obj["source_references"]:

        if not isinstance(
            source_id,
            int
        ):
            raise ValueError(
                "source_references must contain integers."
            )

        if source_id < 1 or source_id > 5:
            raise ValueError(
                "source_references must contain numbers 1-5."
            )

    return True

In [ ]:
def build_prompt(row, rag_chunks):

    predicted_label = int(row["predicted_label"])
    predicted_class = CLASS_NAMES[predicted_label]

    # Find probability columns
    probability_cols = [
        c for c in row.index
        if "prob" in str(c).lower()
    ]

    probabilities = {}

    for c in probability_cols:
        try:
            probabilities[str(c)] = float(row[c])
        except Exception:
            pass

    gradcam_area = row.get(
        "high_activation_area_percent",
        None
    )

    trust_score = float(row["trust_score"])

    # --------------------------------------------------
    # Prepare RAG context
    # --------------------------------------------------

    rag_context = []

    for i, chunk in enumerate(rag_chunks[:5], 1):

        text = str(chunk.get("text", ""))

        # Keep context compact
        if len(text) > 500:
            text = text[:500]

        rag_context.append(
            f"SOURCE {i}: {text}"
        )

    rag_text = "\n".join(rag_context)

    # --------------------------------------------------
    # Prompt
    # --------------------------------------------------

    prompt = f"""
You are an agricultural AI explanation assistant.

Your task is to explain a machine-learning classifier result for an apple leaf.

PREDICTED CLASS:
{predicted_class}

CLASS PROBABILITIES:
{json.dumps(probabilities)}

TRUST SCORE:
{trust_score:.6f}

GRAD-CAM HIGH-ACTIVATION AREA:
{gradcam_area}

RETRIEVED AGRICULTURAL KNOWLEDGE:
{rag_text}

RULES:

1. Do not change the predicted class.

2. Call the result a classifier prediction, not a confirmed diagnosis.

3. Do not claim that you directly see symptoms in the image.

4. Do not say that the leaf is visibly healthy or visibly diseased.

5. Do not invent image symptoms.

6. Grad-CAM identifies image regions that influenced the classifier. It is not ground-truth disease localization.

7. Retrieved knowledge is general agricultural knowledge. It is not proof that a symptom is present in this image.

8. Do not say that other diseases are "ruled out."

9. Trust score represents model reliability or uncertainty, not biological certainty.

10. A large prediction margin means the top prediction is strongly separated from the alternatives.

11. Do not describe a large prediction margin as weak or small uncertainty.

12. Do not provide treatment recommendations.

13. Do not provide URLs.

14. Use source numbers only.

15. Keep every answer short.

16. Human review should be described neutrally.

17. Do not claim that human review is unnecessary.

18. Do not make absolute claims about correctness.

Return ONLY this JSON object:

{{
  "summary": "one short sentence",
  "prediction_explanation": "one short sentence",
  "visual_evidence": "one short sentence",
  "knowledge_evidence": "one short sentence",
  "uncertainty_note": "one short sentence",
  "human_review_recommendation": "one short sentence",
  "source_references": [1, 2]
}}

The source_references field must contain only integers from 1 to 5.

Do not add any other fields.

Do not use Markdown.

Do not use code fences.

Do not write anything before or after the JSON.
"""

    return prompt

In [ ]:
# ============================================================
# 15. FINAL SMOKE TEST — DEBUG + JSON VALIDATION
# ============================================================

smoke_row = llm_df.iloc[0]

smoke_sample_id = str(
    smoke_row["sample_id"]
)

smoke_class = CLASS_NAMES[
    int(smoke_row["predicted_label"])
]

smoke_rag = rag_cache[
    smoke_class
]

print("==========================================")
print("FINAL SMOKE TEST")
print("==========================================")

print("Sample:", smoke_sample_id)
print("Prediction:", smoke_class)
print(
    "Trust:",
    float(smoke_row["trust_score"])
)

print(
    "RAG chunks:",
    len(smoke_rag)
)

# ------------------------------------------------------------
# BUILD PROMPT
# ------------------------------------------------------------

smoke_prompt = build_prompt(
    smoke_row,
    smoke_rag
)

print("\nGenerating Qwen response...")

# ------------------------------------------------------------
# GENERATE
# ------------------------------------------------------------

smoke_response = generate_qwen(
    smoke_prompt,
    max_new_tokens=250
)

# ------------------------------------------------------------
# ALWAYS SHOW RAW RESPONSE
# ------------------------------------------------------------

print(
    "\n================ RAW QWEN OUTPUT ================\n"
)

print(smoke_response)

print(
    "\n=================================================="
)

print(
    "\nResponse length:",
    len(smoke_response)
)

# ------------------------------------------------------------
# TRY JSON EXTRACTION
# ------------------------------------------------------------

try:

    smoke_json = clean_json_response(
        smoke_response
    )

    validate_json(
        smoke_json
    )

    print(
        "\n=========================================="
    )

    print(
        "✅ SMOKE TEST JSON VALIDATION PASSED"
    )

    print(
        "=========================================="
    )

    print(
        json.dumps(
            smoke_json,
            indent=2,
            ensure_ascii=False
        )
    )

except Exception as e:

    print(
        "\n=========================================="
    )

    print(
        "❌ JSON VALIDATION FAILED"
    )

    print(
        "=========================================="
    )

    print(
        "Error:",
        e
    )

    print(
        "\nIMPORTANT: The RAW QWEN OUTPUT above "
        "is what we need to inspect."
    )

    raise

In [ ]:
# 16. SAVE SMOKE TEST RESULT TO DRIVE

smoke_output = {
    "sample_id": smoke_sample_id,
    "predicted_label": int(
        smoke_row["predicted_label"]
    ),
    "predicted_class": smoke_class,
    "trust_score": float(
        smoke_row["trust_score"]
    ),
    "trust_level": str(
        smoke_row["trust_level"]
    ),
    "gradcam_high_activation_area_percent":
        float(
            smoke_row[
                "high_activation_area_percent"
            ]
        ),
    "classifier_probabilities":
        probabilities(smoke_row),
    "rag_evidence": smoke_rag,
    "llm_explanation": smoke_json,
}

SMOKE_FILE = (
    LLM_30_DIR /
    "smoke_test_result.json"
)

with open(
    SMOKE_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        smoke_output,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Smoke test saved:")
print(SMOKE_FILE)

In [ ]:
# 17. FINAL READINESS CHECK

checks = {
    "30 samples selected": len(llm_df) == 30,
    "RAG cache available": rag_cache is not None,
    "RAG cache has all 4 classes":
        all(
            d in rag_cache
            for d in CLASS_NAMES.values()
        ),
    "Qwen model loaded":
        "llm_model" in globals(),
    "JSON extractor available":
        "clean_json_response" in globals(),
    "JSON validator available":
        "validate_json" in globals(),
    "Smoke test passed":
        "smoke_json" in globals(),
}

for name, status in checks.items():
    print(
        f"{'✅' if status else '❌'} {name}"
    )

if not all(checks.values()):
    raise RuntimeError(
        "Readiness check failed."
    )

print("\n✅ READY TO RUN 30 SAMPLES.")

In [ ]:
# ============================================================
# CELL — RUN QWEN ON 30 BALANCED SAMPLES
# Resumable + Retry + Immediate Saving
# ============================================================

import os
import json
import time
import traceback
from pathlib import Path
from tqdm.auto import tqdm

# ------------------------------------------------------------
# OUTPUT DIRECTORIES
# ------------------------------------------------------------

LLM_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/XAI_Project/results/llm_explanations/llm_30_samples"
)

PER_SAMPLE_DIR = LLM_OUTPUT_DIR / "per_sample"

LLM_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PER_SAMPLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ERROR_FILE = (
    LLM_OUTPUT_DIR /
    "llm_errors_30_samples.jsonl"
)

# ------------------------------------------------------------
# GROUNDING SANITIZER
# ------------------------------------------------------------

def sanitize_llm_output(obj):

    # Prevent unsupported image-specific claims
    obj["visual_evidence"] = (
        "Grad-CAM identifies image regions that "
        "influenced the classifier prediction; "
        "it is not ground-truth disease localization."
    )

    # Prevent RAG knowledge from being treated as
    # image-specific evidence
    obj["knowledge_evidence"] = (
        "Retrieved agricultural knowledge provides "
        "general information about the predicted and "
        "related classes; it does not establish "
        "image-specific symptoms."
    )

    # Keep human review recommendation neutral
    obj["human_review_recommendation"] = (
        "Human review can be used to independently "
        "verify the classifier prediction when appropriate."
    )

    return obj


# ------------------------------------------------------------
# VALIDATE + SANITIZE
# ------------------------------------------------------------

def process_llm_response(response):

    # Extract JSON
    obj = clean_json_response(response)

    # Validate structure
    validate_json(obj)

    # Sanitize unsupported claims
    obj = sanitize_llm_output(obj)

    # Validate again
    validate_json(obj)

    return obj


# ------------------------------------------------------------
# RETRY FUNCTION
# ------------------------------------------------------------

def generate_with_retry(
    prompt,
    max_new_tokens=160,
    retries=1
):

    last_error = None
    last_response = ""

    for attempt in range(retries + 1):

        try:

            # First attempt
            if attempt == 0:

                response = generate_qwen(
                    prompt,
                    max_new_tokens=max_new_tokens
                )

            # Retry with an even stricter instruction
            else:

                retry_prompt = prompt + """

IMPORTANT:
Your previous response may have been incomplete.

Return ONLY the complete JSON object.

Use extremely short sentences.

The response MUST end with:
}

Do not stop before completing the JSON.
"""

                response = generate_qwen(
                    retry_prompt,
                    max_new_tokens=180
                )

            last_response = response

            result = process_llm_response(
                response
            )

            return result, response, None

        except Exception as e:

            last_error = str(e)

            if attempt < retries:

                print(
                    f"   ⚠️ Attempt {attempt + 1} failed. "
                    f"Retrying..."
                )

                time.sleep(1)

    return (
        None,
        last_response,
        last_error
    )


# ------------------------------------------------------------
# ERROR LOGGER
# ------------------------------------------------------------

def save_error(
    sample_id,
    error,
    raw_response=""
):

    error_record = {
        "sample_id": str(sample_id),
        "error": str(error),
        "raw_response": str(raw_response),
        "timestamp": time.strftime(
            "%Y-%m-%d %H:%M:%S"
        )
    }

    with open(
        ERROR_FILE,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            json.dumps(
                error_record,
                ensure_ascii=False
            ) + "\n"
        )


# ------------------------------------------------------------
# CHECK DATA
# ------------------------------------------------------------

assert len(llm_df) == 30, (
    f"Expected 30 samples, found {len(llm_df)}"
)

print("==========================================")
print("30-SAMPLE QWEN INFERENCE")
print("==========================================")

print(
    "Total samples:",
    len(llm_df)
)

print(
    "Output directory:",
    LLM_OUTPUT_DIR
)

# ------------------------------------------------------------
# COUNT ALREADY COMPLETED
# ------------------------------------------------------------

completed_ids = set()

for json_file in PER_SAMPLE_DIR.glob(
    "*.json"
):

    completed_ids.add(
        json_file.stem
    )

print(
    "Already completed:",
    len(completed_ids)
)

remaining = [
    row
    for _, row in llm_df.iterrows()
    if str(row["sample_id"])
    not in completed_ids
]

print(
    "Remaining:",
    len(remaining)
)

print("==========================================")

In [ ]:
# ============================================================
# CELL — FINAL LLM INFERENCE HELPERS
# ============================================================

import os
import json
import time
import traceback
from pathlib import Path

# ------------------------------------------------------------
# OUTPUT PATHS
# ------------------------------------------------------------

LLM_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/XAI_Project/results/llm_explanations/llm_30_samples"
)

PER_SAMPLE_DIR = LLM_OUTPUT_DIR / "per_sample"

ERROR_FILE = (
    LLM_OUTPUT_DIR /
    "llm_errors_30_samples.jsonl"
)

LLM_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PER_SAMPLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# SANITIZE LLM OUTPUT
# ------------------------------------------------------------

def sanitize_llm_output(
    obj,
    rag_chunks=None
):

    # --------------------------------------------------------
    # Grad-CAM grounding
    # --------------------------------------------------------

    obj["visual_evidence"] = (
        "Grad-CAM identifies image regions that "
        "influenced the classifier prediction; "
        "it is not ground-truth disease localization."
    )

    # --------------------------------------------------------
    # RAG grounding
    # --------------------------------------------------------

    obj["knowledge_evidence"] = (
        "Retrieved agricultural knowledge provides "
        "general information about the predicted and "
        "related classes; it does not establish "
        "image-specific symptoms."
    )

    # --------------------------------------------------------
    # Human review
    # --------------------------------------------------------

    obj["human_review_recommendation"] = (
        "Human review can be used to independently "
        "verify the classifier prediction when appropriate."
    )

    # --------------------------------------------------------
    # Source references
    #
    # IMPORTANT:
    # We generate these ourselves from the actual RAG
    # retrieval instead of trusting Qwen to generate them.
    # --------------------------------------------------------

    if rag_chunks is not None:

        obj["source_references"] = list(
            range(
                1,
                min(len(rag_chunks), 5) + 1
            )
        )

    return obj


# ------------------------------------------------------------
# GENERATE + PARSE + VALIDATE + RETRY
# ------------------------------------------------------------

def generate_with_retry(
    prompt,
    rag_chunks,
    max_new_tokens=160,
    retries=1
):

    last_error = None
    last_response = ""

    for attempt in range(
        retries + 1
    ):

        try:

            # ------------------------------------------------
            # FIRST GENERATION
            # ------------------------------------------------

            if attempt == 0:

                response = generate_qwen(
                    prompt,
                    max_new_tokens=max_new_tokens
                )

            # ------------------------------------------------
            # RETRY
            # ------------------------------------------------

            else:

                retry_prompt = prompt + """

IMPORTANT:
Return the complete JSON object.
Use very short sentences.
Do not add explanations.
Do not add URLs.
Do not add Markdown.
Do not add code fences.
Make sure every required field is present.
End with the closing } character.
"""

                response = generate_qwen(
                    retry_prompt,
                    max_new_tokens=180
                )

            last_response = response

            # ------------------------------------------------
            # PARSE JSON
            # ------------------------------------------------

            result = clean_json_response(
                response
            )

            # ------------------------------------------------
            # VALIDATE ORIGINAL JSON
            # ------------------------------------------------

            validate_json(
                result
            )

            # ------------------------------------------------
            # SANITIZE
            # ------------------------------------------------

            result = sanitize_llm_output(
                result,
                rag_chunks
            )

            # ------------------------------------------------
            # VALIDATE FINAL JSON
            # ------------------------------------------------

            validate_json(
                result
            )

            return (
                result,
                response,
                None
            )

        except Exception as e:

            last_error = str(e)

            print(
                f"   ⚠️ Attempt {attempt + 1} failed: "
                f"{e}"
            )

            # Show output only when the first attempt fails
            # so we can diagnose future issues.
            if attempt == 0:

                if last_response:

                    print(
                        "   Raw output preview:"
                    )

                    print(
                        last_response[:500]
                    )

            if attempt < retries:

                print(
                    "   Retrying..."
                )

                time.sleep(1)

    return (
        None,
        last_response,
        last_error
    )


# ------------------------------------------------------------
# SAVE ERROR
# ------------------------------------------------------------

def save_error(
    sample_id,
    error,
    raw_response=""
):

    error_record = {

        "sample_id":
            str(sample_id),

        "error":
            str(error),

        "raw_response":
            str(raw_response),

        "timestamp":
            time.strftime(
                "%Y-%m-%d %H:%M:%S"
            )
    }

    with open(
        ERROR_FILE,
        "a",
        encoding="utf-8"
    ) as f:

        f.write(
            json.dumps(
                error_record,
                ensure_ascii=False
            ) + "\n"
        )


print("==========================================")
print("LLM HELPER FUNCTIONS READY")
print("==========================================")

print(
    "Output directory:",
    LLM_OUTPUT_DIR
)

print(
    "Per-sample directory:",
    PER_SAMPLE_DIR
)

print("✅ Helpers loaded successfully.")

In [ ]:
# ============================================================
# CELL — FINAL 30-SAMPLE LLM INFERENCE
# Resumable + Retry + Immediate Saving
# ============================================================

from tqdm.auto import tqdm

# ------------------------------------------------------------
# BASIC CHECKS
# ------------------------------------------------------------

if "llm_df" not in globals():

    raise RuntimeError(
        "llm_df is not loaded. "
        "Run the sample-selection cell first."
    )

if "rag_cache" not in globals():

    raise RuntimeError(
        "rag_cache is not loaded. "
        "Run the RAG cache cell first."
    )

if "llm_model" not in globals():

    raise RuntimeError(
        "Qwen model is not loaded. "
        "Run the Qwen model-loading cell first."
    )

# ------------------------------------------------------------
# CHECK 30 SAMPLES
# ------------------------------------------------------------

if len(llm_df) != 30:

    raise ValueError(
        f"Expected exactly 30 samples, "
        f"but found {len(llm_df)}."
    )

# ------------------------------------------------------------
# OUTPUT PATHS
# ------------------------------------------------------------

LLM_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/XAI_Project/results/llm_explanations/llm_30_samples"
)

PER_SAMPLE_DIR = (
    LLM_OUTPUT_DIR /
    "per_sample"
)

LLM_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PER_SAMPLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ERROR_FILE = (
    LLM_OUTPUT_DIR /
    "llm_errors_30_samples.jsonl"
)

# ------------------------------------------------------------
# FIND ALREADY COMPLETED SAMPLES
# ------------------------------------------------------------

completed_ids = set()

for json_file in PER_SAMPLE_DIR.glob(
    "*.json"
):

    try:

        with open(
            json_file,
            "r",
            encoding="utf-8"
        ) as f:

            existing = json.load(f)

        # Only count valid completed files
        if (
            isinstance(existing, dict)
            and
            "sample_id" in existing
            and
            "llm_explanation" in existing
        ):

            completed_ids.add(
                str(existing["sample_id"])
            )

    except Exception:

        # Ignore corrupted/incomplete files.
        pass


# ------------------------------------------------------------
# DETERMINE REMAINING SAMPLES
# ------------------------------------------------------------

remaining_rows = []

for _, row in llm_df.iterrows():

    sample_id = str(
        row["sample_id"]
    )

    if sample_id not in completed_ids:

        remaining_rows.append(
            row
        )

# ------------------------------------------------------------
# STATUS
# ------------------------------------------------------------

print("==========================================")
print("FINAL 30-SAMPLE LLM INFERENCE")
print("==========================================")

print(
    "Total selected samples:",
    len(llm_df)
)

print(
    "Already completed:",
    len(completed_ids)
)

print(
    "Remaining:",
    len(remaining_rows)
)

print(
    "Output:",
    PER_SAMPLE_DIR
)

print("==========================================")

# ------------------------------------------------------------
# IF EVERYTHING IS ALREADY DONE
# ------------------------------------------------------------

if len(remaining_rows) == 0:

    print(
        "\n✅ All 30 samples are already completed."
    )

else:

    print(
        "\nStarting inference..."
    )

    # --------------------------------------------------------
    # MAIN LOOP
    # --------------------------------------------------------

    for local_idx, row in enumerate(
        remaining_rows,
        start=1
    ):

        sample_id = str(
            row["sample_id"]
        )

        predicted_label = int(
            row["predicted_label"]
        )

        predicted_class = CLASS_NAMES[
            predicted_label
        ]

        trust_score = float(
            row["trust_score"]
        )

        output_file = (
            PER_SAMPLE_DIR /
            f"{sample_id}.json"
        )

        print("\n------------------------------------------")

        print(
            f"[{local_idx}/{len(remaining_rows)}] "
            f"{sample_id}"
        )

        print(
            "Prediction:",
            predicted_class
        )

        print(
            f"Trust: {trust_score:.6f}"
        )

        try:

            # ------------------------------------------------
            # GET RAG RESULTS
            # ------------------------------------------------

            if predicted_class not in rag_cache:

                raise ValueError(
                    f"RAG cache missing for "
                    f"class: {predicted_class}"
                )

            rag_chunks = rag_cache[
                predicted_class
            ]

            # ------------------------------------------------
            # BUILD PROMPT
            # ------------------------------------------------

            prompt = build_prompt(
                row,
                rag_chunks
            )

            # ------------------------------------------------
            # GENERATE
            # ------------------------------------------------

            result, raw_response, error = (
                generate_with_retry(
                    prompt,
                    rag_chunks,
                    max_new_tokens=160,
                    retries=1
                )
            )

            # ------------------------------------------------
            # FAILURE
            # ------------------------------------------------

            if result is None:

                print(
                    f"   ❌ FAILED: {error}"
                )

                save_error(
                    sample_id,
                    error,
                    raw_response
                )

                continue

            # ------------------------------------------------
            # GRAD-CAM VALUE
            # ------------------------------------------------

            gradcam_value = None

            if (
                "high_activation_area_percent"
                in row.index
            ):

                try:

                    value = float(
                        row[
                            "high_activation_area_percent"
                        ]
                    )

                    if not (
                        pd.isna(value)
                    ):

                        gradcam_value = value

                except Exception:

                    gradcam_value = None

            # ------------------------------------------------
            # CREATE FINAL RECORD
            # ------------------------------------------------

            result_record = {

                "sample_id":
                    sample_id,

                "predicted_label":
                    predicted_label,

                "predicted_class":
                    predicted_class,

                "trust_score":
                    trust_score,

                "gradcam_high_activation_area_percent":
                    gradcam_value,

                "llm_explanation":
                    result,

                "rag_sources_used":
                    [
                        str(
                            chunk.get(
                                "source",
                                f"Source {i + 1}"
                            )
                        )

                        for i, chunk
                        in enumerate(
                            rag_chunks[:5]
                        )
                    ]
            }

            # ------------------------------------------------
            # SAVE IMMEDIATELY
            # ------------------------------------------------

            temp_file = (
                PER_SAMPLE_DIR /
                f"{sample_id}.tmp.json"
            )

            with open(
                temp_file,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump(
                    result_record,
                    f,
                    indent=2,
                    ensure_ascii=False
                )

            # Atomic-ish rename after successful write
            temp_file.replace(
                output_file
            )

            # ------------------------------------------------
            # SUCCESS
            # ------------------------------------------------

            print(
                "   ✅ Saved successfully"
            )

            print(
                "   Summary:",
                result.get(
                    "summary",
                    ""
                )
            )

        except Exception as e:

            print(
                f"   ❌ ERROR: {e}"
            )

            save_error(
                sample_id,
                traceback.format_exc(),
                ""
            )

        # ----------------------------------------------------
        # SMALL PAUSE
        # ----------------------------------------------------

        time.sleep(0.5)


# ============================================================
# FINAL STATUS
# ============================================================

final_json_files = list(
    PER_SAMPLE_DIR.glob(
        "*.json"
    )
)

print("\n==========================================")
print("LLM INFERENCE STATUS")
print("==========================================")

print(
    "Expected samples:",
    len(llm_df)
)

print(
    "Saved sample files:",
    len(final_json_files)
)

# ------------------------------------------------------------
# CLASS DISTRIBUTION
# ------------------------------------------------------------

class_counts = {}

for json_file in final_json_files:

    try:

        with open(
            json_file,
            "r",
            encoding="utf-8"
        ) as f:

            record = json.load(f)

        cls = record.get(
            "predicted_class",
            "Unknown"
        )

        class_counts[cls] = (
            class_counts.get(
                cls,
                0
            ) + 1
        )

    except Exception:

        pass

print("\nClass distribution:")

for cls, count in class_counts.items():

    print(
        f"  {cls}: {count}"
    )

# ------------------------------------------------------------
# COMPLETION CHECK
# ------------------------------------------------------------

if len(final_json_files) == 30:

    print(
        "\n✅ ALL 30 LLM EXPLANATIONS SAVED."
    )

else:

    print(
        f"\n⚠️ {30 - len(final_json_files)} "
        f"samples are still incomplete."
    )

print(
    "\nPer-sample directory:"
)

print(
    PER_SAMPLE_DIR
)

print("==========================================")

In [ ]:
# ============================================================
# CELL — CONSOLIDATE 30 LLM RESULTS
# JSON files → JSONL + CSV
# ============================================================

import os
import json
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

LLM_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/XAI_Project/results/llm_explanations/llm_30_samples"
)

PER_SAMPLE_DIR = (
    LLM_OUTPUT_DIR / "per_sample"
)

JSONL_FILE = (
    LLM_OUTPUT_DIR /
    "llm_explanations_30_samples.jsonl"
)

CSV_FILE = (
    LLM_OUTPUT_DIR /
    "llm_explanations_30_samples.csv"
)

# Make sure directories exist
LLM_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

PER_SAMPLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# FIND PER-SAMPLE FILES
# ------------------------------------------------------------

json_files = sorted(
    PER_SAMPLE_DIR.glob("*.json")
)

print("==========================================")
print("CONSOLIDATING LLM RESULTS")
print("==========================================")

print(
    "Per-sample JSON files found:",
    len(json_files)
)

if len(json_files) == 0:

    raise FileNotFoundError(
        f"No JSON files found in:\n{PER_SAMPLE_DIR}"
    )

# ------------------------------------------------------------
# LOAD VALID JSON FILES
# ------------------------------------------------------------

records = []
invalid_files = []

for json_file in json_files:

    try:

        with open(
            json_file,
            "r",
            encoding="utf-8"
        ) as f:

            record = json.load(f)

        # Basic validation
        if not isinstance(
            record,
            dict
        ):
            raise ValueError(
                "JSON root is not a dictionary."
            )

        if "sample_id" not in record:
            raise ValueError(
                "Missing sample_id."
            )

        if "llm_explanation" not in record:
            raise ValueError(
                "Missing llm_explanation."
            )

        # Validate LLM explanation
        validate_json(
            record["llm_explanation"]
        )

        records.append(
            record
        )

    except Exception as e:

        print(
            f"❌ Invalid file: "
            f"{json_file.name}"
        )

        print(
            f"   Error: {e}"
        )

        invalid_files.append(
            json_file.name
        )

# ------------------------------------------------------------
# WRITE JSONL
# ------------------------------------------------------------

with open(
    JSONL_FILE,
    "w",
    encoding="utf-8"
) as f:

    for record in records:

        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

# ------------------------------------------------------------
# CREATE FLAT CSV
# ------------------------------------------------------------

csv_rows = []

for record in records:

    explanation = record.get(
        "llm_explanation",
        {}
    )

    csv_rows.append({

        "sample_id":
            record.get(
                "sample_id"
            ),

        "predicted_label":
            record.get(
                "predicted_label"
            ),

        "predicted_class":
            record.get(
                "predicted_class"
            ),

        "trust_score":
            record.get(
                "trust_score"
            ),

        "gradcam_high_activation_area_percent":
            record.get(
                "gradcam_high_activation_area_percent"
            ),

        "summary":
            explanation.get(
                "summary"
            ),

        "prediction_explanation":
            explanation.get(
                "prediction_explanation"
            ),

        "visual_evidence":
            explanation.get(
                "visual_evidence"
            ),

        "knowledge_evidence":
            explanation.get(
                "knowledge_evidence"
            ),

        "uncertainty_note":
            explanation.get(
                "uncertainty_note"
            ),

        "human_review_recommendation":
            explanation.get(
                "human_review_recommendation"
            ),

        "source_references":
            json.dumps(
                explanation.get(
                    "source_references",
                    []
                )
            ),

        "rag_sources_used":
            json.dumps(
                record.get(
                    "rag_sources_used",
                    []
                ),
                ensure_ascii=False
            )
    })

# ------------------------------------------------------------
# SAVE CSV
# ------------------------------------------------------------

df_llm = pd.DataFrame(
    csv_rows
)

df_llm.to_csv(
    CSV_FILE,
    index=False,
    encoding="utf-8"
)

# ------------------------------------------------------------
# FINAL REPORT
# ------------------------------------------------------------

print("\n==========================================")
print("CONSOLIDATION COMPLETE")
print("==========================================")

print(
    "Valid samples:",
    len(records)
)

print(
    "Invalid samples:",
    len(invalid_files)
)

print(
    "\nJSONL:"
)

print(
    JSONL_FILE
)

print(
    "\nCSV:"
)

print(
    CSV_FILE
)

if invalid_files:

    print(
        "\n⚠️ Invalid files:"
    )

    for filename in invalid_files:
        print(
            " -",
            filename
        )

else:

    print(
        "\n✅ All per-sample JSON files are valid."
    )

# ------------------------------------------------------------
# DISPLAY PREVIEW
# ------------------------------------------------------------

if len(df_llm) > 0:

    print(
        "\n=========================================="
    )

    print(
        "CSV PREVIEW"
    )

    print(
        "=========================================="
    )

    display(
        df_llm.head()
    )

print(
    "\nFiles saved successfully."
)

In [ ]:
# ============================================================
# CELL — SAVE FINAL NOTEBOOK 6 CONFIGURATION
# ============================================================

import json
from pathlib import Path
from datetime import datetime

# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------

LLM_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/XAI_Project/results/llm_explanations/llm_30_samples"
)

LLM_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CONFIG_FILE = (
    LLM_OUTPUT_DIR /
    "llm_30_config.json"
)

# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

MAX_NEW_TOKENS = 160
RETRY_MAX_NEW_TOKENS = 180
NUM_SAMPLES = 30
TOP_K = 5

config = {

    "notebook":
        "06_Final_LLM_30_Samples_Qwen_RAG",

    "project":
        "Explainable Apple Plant Disease Detection",

    "created_at":
        datetime.now().isoformat(),

    # --------------------------------------------------------
    # Classifier
    # --------------------------------------------------------

    "classifier":
        "ResNet50",

    "num_classes":
        4,

    "class_names":
        CLASS_NAMES,

    "input_shape":
        [3, 224, 224],

    # --------------------------------------------------------
    # Trust / Uncertainty
    # --------------------------------------------------------

    "trust_score_used":
        True,

    "trust_score_role":
        "Model reliability and uncertainty signal",

    # --------------------------------------------------------
    # Explainability
    # --------------------------------------------------------

    "gradcam_used":
        True,

    "gradcam_target_layer":
        "model.layer4[-1]",

    "gradcam_role":
        "Model-attention evidence, not ground-truth localization",

    # --------------------------------------------------------
    # RAG
    # --------------------------------------------------------

    "rag_used":
        True,

    "rag_top_k":
        TOP_K,

    "rag_index":
        "apple_disease_faiss.index",

    "rag_metadata":
        "knowledge_chunks.csv",

    "embedding_model":
        "sentence-transformers/all-MiniLM-L6-v2",

    "embedding_dimension":
        384,

    # --------------------------------------------------------
    # LLM
    # --------------------------------------------------------

    "llm_model":
        "Qwen/Qwen2.5-3B-Instruct",

    "quantization":
        "4-bit NF4",

    "max_new_tokens":
        MAX_NEW_TOKENS,

    "retry_max_new_tokens":
        RETRY_MAX_NEW_TOKENS,

    "do_sample":
        False,

    "num_beams":
        1,

    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    "llm_evaluation_samples":
        NUM_SAMPLES,

    "sampling_strategy":
        "Balanced class sampling",

    "sample_distribution":
        {
            "Apple Healthy": 8,
            "Apple Scab": 8,
            "Apple Black Rot": 7,
            "Apple Cedar Apple Rust": 7
        },

    # --------------------------------------------------------
    # Output schema
    # --------------------------------------------------------

    "json_fields":
        [
            "summary",
            "prediction_explanation",
            "visual_evidence",
            "knowledge_evidence",
            "uncertainty_note",
            "human_review_recommendation",
            "source_references"
        ],

    # --------------------------------------------------------
    # Reliability
    # --------------------------------------------------------

    "retry_enabled":
        True,

    "max_retries":
        1,

    "immediate_per_sample_saving":
        True,

    "resumable_inference":
        True,

    # --------------------------------------------------------
    # Grounding
    # --------------------------------------------------------

    "grounding_rules":
        [
            "LLM does not change classifier prediction",
            "LLM does not provide confirmed diagnosis",
            "Grad-CAM is not ground-truth localization",
            "RAG knowledge is general knowledge",
            "RAG knowledge is not proof of image-specific symptoms",
            "Trust score is not biological certainty",
            "Human review recommendation is neutral",
            "Source references are derived from actual RAG retrieval"
        ]
}

# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

with open(
    CONFIG_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        config,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

print("==========================================")
print("NOTEBOOK 6 CONFIGURATION SAVED")
print("==========================================")

print(
    "Config file:"
)

print(
    CONFIG_FILE
)

print(
    "\nMax new tokens:",
    MAX_NEW_TOKENS
)

print(
    "Retry max tokens:",
    RETRY_MAX_NEW_TOKENS
)

print(
    "LLM samples:",
    NUM_SAMPLES
)

print(
    "RAG top-k:",
    TOP_K
)

print(
    "\n✅ Configuration saved successfully."
)

# Notebook 6 — Final Summary

## What this notebook does

This notebook adds the **grounded LLM explanation layer** to the Apple Plant Disease XAI pipeline.

The classifier and quantitative XAI evaluation have already been completed on the full **1,166-image test set**.

This notebook therefore focuses only on producing a manageable, balanced set of **30 grounded natural-language explanations**.

## Architecture

```text
Full test set: 1166 images
        │
        ├── ResNet50 prediction
        ├── class probabilities
        ├── Trust / uncertainty
        └── Grad-CAM
                │
                ▼
        Select 30 balanced samples
        8 Healthy
        8 Scab
        7 Black Rot
        7 Cedar Apple Rust
                │
                ▼
        FAISS RAG retrieval
                │
                ▼
        Qwen2.5-3B-Instruct
        4-bit NF4
                │
                ▼
        Structured JSON explanation
                │
                ▼
        Google Drive
```

## Components

### Classifier
- ResNet50
- Input: `3 × 224 × 224`
- Output: 4 disease classes

### Trust / uncertainty
Notebook 4 provides:
- confidence
- entropy
- margin
- trust score
- trust level

### Visual explanation
Grad-CAM provides:
- model-attention information
- high-activation area percentage

Grad-CAM is treated as an explanation signal, **not ground-truth localization**.

### RAG
Notebook 5 provides:
- FAISS index
- 384-dimensional MiniLM embeddings
- `knowledge_chunks.csv`
- top-5 retrieved knowledge chunks

MiniLM runs on CPU in this notebook so that the T4 GPU remains available for Qwen.

### LLM
- Qwen2.5-3B-Instruct
- 4-bit NF4 quantization
- deterministic generation
- maximum 250 generated tokens

The LLM does not replace the classifier. It converts structured evidence into a grounded explanation.

## JSON output

Each sample contains:

- classifier prediction
- probabilities
- confidence
- entropy
- margin
- trust score
- trust level
- Grad-CAM statistic
- retrieved RAG evidence
- summary
- prediction explanation
- visual evidence
- knowledge evidence
- uncertainty note
- human-review recommendation
- source references

## Why only 30 LLM samples?

Generating an LLM explanation for every test image is computationally expensive and produces substantial redundancy when the classifier predictions are already available for all 1,166 samples.

The full 1,166-sample quantitative evaluation remains intact.

The 30 LLM samples provide a balanced qualitative demonstration of the complete:

**Prediction → Trust → Grad-CAM → RAG → LLM → Human Review**

pipeline.

## Saved artifacts

```text
XAI_Project/results/llm_explanations/llm_30_samples/

├── per_sample/
│   ├── test_XXXXX.json
│   └── ...
│
├── smoke_test_result.json
├── rag_retrieval_cache.json
├── llm_explanations_30_samples.jsonl
├── llm_explanations_30_samples.csv
├── llm_30_config.json
└── llm_errors_30_samples.jsonl
```

Every successful sample is saved immediately, making the experiment resumable.

## Important limitations

1. Grad-CAM is not ground-truth localization because the dataset does not contain disease masks.
2. Trust score is a reliability indicator, not a guarantee of correctness.
3. RAG explanations depend on the quality and coverage of the curated knowledge base.
4. LLM explanations are generated from supplied evidence and should not be treated as independent diagnoses.
5. The unusually high classifier performance should be checked for duplicate/near-duplicate images or split leakage before making strong real-world generalization claims.

## Next stage

**Notebook 7 — Human-in-the-Loop + Feedback/Memory**

will use the saved 30 explanations to capture human feedback, corrections/confirmation, and memory for the final integrated XAI system.